# L04 -- Unbalanced (Three-phase) Power Flow

Companion notebook for L04. Reproduces:
- The Fortescue transform in numpy
- The 4-step hand math (currents -> sequence -> phase voltages) for the slide's worked example
- pandapower's converged answer and the linear-vs-converged gap
- VUF in both linear and converged forms
- The five project violation types in code form

**Prerequisite**: L01, L03. Slides: `L04_unbalanced_pf.pdf`.

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import pandapower as pp
from packaging.version import Version
import matplotlib.pyplot as plt

print(f"pandapower {pp.__version__}, numpy {np.__version__}, pandas {pd.__version__}")
assert Version(pp.__version__) >= Version("3.2.1"), "Need pandapower >= 3.2.1 (env: pdpower)"

## 1. The Fortescue transform in numpy

In [ ]:
a = np.exp(2j*np.pi/3)
A = np.array([[1, 1,    1],
              [1, a**2, a],
              [1, a,    a**2]], dtype=complex)
A_inv = np.array([[1, 1,    1],
                  [1, a,    a**2],
                  [1, a**2, a]], dtype=complex) / 3

print("|A * A_inv - I| =", np.abs(A @ A_inv - np.eye(3)).max())
assert np.allclose(A @ A_inv, np.eye(3), atol=1e-12), "Fortescue inverse broken"

# Quick sanity: a single-phase phasor decomposes into equal seq components
v_phase = np.array([1.0+0j, 0, 0])
v_seq = A_inv @ v_phase
print("Sequence components of [1, 0, 0] (zero, positive, negative):", v_seq.round(4))

## 2. Worked example: hand math (linear, flat start)

2-bus LV feeder, $Z^{(1)} = Z^{(2)} = 0.05+j0.05$, $Z^{(0)} = 0.10+j0.10$ pu. Single-phase load $S_a = 0.4$ pu on phase A.

In [ ]:
Z1 = 0.05 + 0.05j
Z2 = 0.05 + 0.05j
Z0 = 0.10 + 0.10j
V_slack = 1.0 + 0j

# Step 1: phase currents under flat-start linear approximation
S_phase = np.array([0.4, 0, 0], dtype=complex)
V_flat  = np.array([1.0, 1.0, 1.0], dtype=complex)
I_phase = np.conj(S_phase / V_flat)
I_seq   = A_inv @ I_phase
print("Sequence currents (I0, I1, I2):", I_seq.round(4))

# Step 2: sequence voltage drops
V0_bus2 = 0       - Z0 * I_seq[0]
V1_bus2 = V_slack - Z1 * I_seq[1]
V2_bus2 = 0       - Z2 * I_seq[2]
V_seq_bus2 = np.array([V0_bus2, V1_bus2, V2_bus2])
print("Sequence voltages at bus 2:", V_seq_bus2.round(4))

# Step 3: phase voltages via inverse Fortescue
V_phase_bus2 = A @ V_seq_bus2
print("Phase voltages at bus 2 (linear hand math):")
for phi, v in zip("abc", V_phase_bus2):
    print(f"  V_{phi} = {v.real:+.4f}{v.imag:+.4f}j   |V_{phi}| = {abs(v):.4f}")

## 3. Same setup in pandapower (constant-power, iterated)

The slides show that the linear hand math under-estimates the violation because constant-power loads amplify it. Verify.

In [ ]:
ZB = 0.16
net = pp.create_empty_network(sn_mva=1.0)
b1 = pp.create_bus(net, vn_kv=0.4); b2 = pp.create_bus(net, vn_kv=0.4)
pp.create_ext_grid(net, bus=b1, vm_pu=1.0,
    s_sc_max_mva=1e6, rx_max=0.1, x0x_max=1.0, r0x0_max=0.1)
pp.create_line_from_parameters(net, b1, b2, length_km=1.0,
    r_ohm_per_km=0.05*ZB, x_ohm_per_km=0.05*ZB,
    r0_ohm_per_km=0.10*ZB, x0_ohm_per_km=0.10*ZB,
    c_nf_per_km=0.0, c0_nf_per_km=0.0, max_i_ka=10.0)
pp.create_asymmetric_load(net, bus=b2, p_a_mw=0.4, q_a_mvar=0.0,
    p_b_mw=0.0, q_b_mvar=0.0, p_c_mw=0.0, q_c_mvar=0.0, type="wye")
pp.runpp_3ph(net)

r = net.res_bus_3ph.iloc[1]
print(f"pandapower converged: V_a={r.vm_a_pu:.4f}, V_b={r.vm_b_pu:.4f}, "
      f"V_c={r.vm_c_pu:.4f}, VUF={r.unbalance_percent:.2f}%")

# Compare
print()
print("Side-by-side:")
print(f"               linear hand-math   pandapower converged")
for phi, v_hand, v_pp in zip("abc",
                             [abs(V_phase_bus2[0]), abs(V_phase_bus2[1]), abs(V_phase_bus2[2])],
                             [r.vm_a_pu, r.vm_b_pu, r.vm_c_pu]):
    print(f"  |V_{phi}|        {v_hand:.4f}             {v_pp:.4f}")

## 4. VUF (linear vs converged)

In [ ]:
VUF_linear   = abs(V_seq_bus2[2]) / abs(V_seq_bus2[1]) * 100
VUF_converged = r.unbalance_percent
print(f"VUF_linear    = {VUF_linear:.2f}%  (< 2% in linear approximation)")
print(f"VUF_converged = {VUF_converged:.2f}%  (> 2%, real violation)")

# The five violation types from the slide
def classify_violations(vm_a, vm_b, vm_c, loading_a, loading_b, loading_c, vuf, converged):
    return {
        "phase_under_voltage": min(vm_a, vm_b, vm_c) < 0.95,
        "phase_over_voltage":  max(vm_a, vm_b, vm_c) > 1.05,
        "phase_overload":      max(loading_a, loading_b, loading_c) > 100.0,
        "vuf_excursion":       vuf > 2.0,
        "non_convergence":     not converged,
    }

flags = classify_violations(r.vm_a_pu, r.vm_b_pu, r.vm_c_pu,
                            0, 0, 0,         # ignore loading for this 2-bus example
                            r.unbalance_percent, net.converged)
print("\nViolation flags:")
for k, v in flags.items():
    print(f"  {k:<22} {v}")

## Homework

### Required
1. Replace the single-phase load with an `asymmetric_sgen` of 0.4 pu on phase A. Which phase now sees overvoltage? Recompute the per-phase voltages and VUF.
2. Vary $Z_0/Z_1$ from 1 to 10 and plot $|V_a|$ at bus 2.
3. Move the load from phase A to phase B (single-phase on B). Verify by symmetry which phase voltage now drops most.

### Optional
- Implement a delta-connected asymmetric load and rederive the VUF. Compare.

In [ ]:
# TODO Q1: PV instead of load
# ...

# TODO Q2: Z_0/Z_1 sweep
# ...

# TODO Q3: load on phase B
# ...